## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:

A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
A function or coroutine to execute.

In [11]:
import os

from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

model = init_chat_model("gpt-4.1-mini", max_tokens=30)
response=model.invoke("Why do parrots talk?")
print(response.content)
print(response.response_metadata["token_usage"])

Parrots "talk" — meaning they can mimic human speech sounds — primarily because of their highly developed vocal abilities and social nature.

Here are the main
{'completion_tokens': 30, 'prompt_tokens': 13, 'total_tokens': 43, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}


In [8]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """
    Get's the weather at a location
    """
    return f"It is sunny in: {location}"

model_with_tools = model.bind_tools([get_weather])

In [16]:
response = model_with_tools.invoke("What is the weather at New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"tool name is: {tool_call['name']}")
    print(f"tool args are: {tool_call['args']}")
    tool_result = get_weather.invoke(tool_call)
    print(tool_result)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 53, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_caf3f6485e', 'id': 'chatcmpl-DmevbjV4vuSiKCdTKAWM7gKdAgKDj', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019e8d70-d69e-7713-9aaf-1ec04a3eb15a-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_DTmDGDkmWpXcifABbsB3FeQY', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 15, 'total_tokens': 68, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
tool name is: get_weathe

### Tool execution loops

In [17]:
# Step 1: Model generates tool calls
messages = [
    {"role": "user", "content": "What's the weather in Boston?"},
]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

The weather in Boston is sunny. Is there anything else you'd like to know?


In [18]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 51, 'total_tokens': 65, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_caf3f6485e', 'id': 'chatcmpl-DmezDwoTLHIn3mJL2X9LJgcR2Fqrd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e8d74-3f3a-7880-8b43-c72b1a13652b-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_usz5SI6YS4jqKv4tqgyopsLa', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 14, 'total_tokens': 65, 'input_token_details': {'audio': 0, 'cache_read': 0}, 